# Author: Michael Chen (mwc2150@columbia.edu)

# 0. Introduction

This notebook answers the core question: **"Has Everlane's sustainability messaging become more concrete over time, or has it diverged from public perceptions of ethical practice?"**

It operates on the corpus built in `01_scraping.ipynb` — three sources, three voices:
- **Reddit** (consumer sentiment): 314 posts + 4,429 comments
- **Everlane website** (brand narrative): 31 pages
- **GDELT news** (media framing): 1,015 articles

The analysis runs in two passes:
1. **Manual LLM pipeline** — Claude API on a hand-picked sample to develop qualitative intuitions
2. **Automated BERT pipeline** — HuggingFace NER + sentence-level sentiment scaled across the full corpus

Inter-party valence is the key unit: when Source A mentions Party B, what is the sentiment of *that specific sentence*? This lets us compare how consumers talk about Everlane vs. how Everlane talks about its workers vs. how the press frames the brand.

# 0. Setup

In [ ]:
import sys
!{sys.executable} -m pip install anthropic transformers torch nltk pandas matplotlib seaborn tqdm --quiet

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import nltk
from tqdm.auto import tqdm

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

tqdm.pandas()
pd.options.display.max_colwidth = 120
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Load all four sources
df_posts    = pd.read_csv("../data/raw/reddit_posts.csv", parse_dates=["date"])
df_comments = pd.read_csv("../data/raw/reddit_comments.csv", parse_dates=["date"])
df_web      = pd.read_csv("../data/raw/everlane_web.csv")
df_news     = pd.read_csv("../data/raw/gdelt_articles_usable.csv", parse_dates=["publish_date"])

# Normalise Reddit into a single frame with a unified text column
df_reddit = pd.concat([
    df_posts[["subreddit", "date", "title", "text", "score"]].assign(
        text=lambda d: (d["title"].fillna("") + " " + d["text"].fillna("")).str.strip()
    ).drop(columns=["title"]),
    df_comments[["subreddit", "date", "text", "score"]]
], ignore_index=True).dropna(subset=["text"])
df_reddit = df_reddit[df_reddit["text"].str.strip().ne("")]

print(f"Reddit:  {len(df_reddit):,} documents")
print(f"Web:     {len(df_web):,} pages")
print(f"News:    {len(df_news):,} articles")

## Sentence tokenization

Every downstream step operates on individual sentences rather than full documents. We explode each source into a flat sentences frame with consistent columns: `source`, `date`, `sentence`, plus any source-specific metadata we want to keep.

In [ ]:
def explode_sentences(df, text_col, source_label, date_col=None, extra_cols=None):
    rows = []
    for _, row in df.iterrows():
        text = str(row[text_col])
        sents = sent_tokenize(text)
        for s in sents:
            s = s.strip()
            if len(s) < 20:   # skip noise fragments
                continue
            entry = {"source": source_label, "sentence": s}
            entry["date"] = row[date_col] if date_col else pd.NaT
            if extra_cols:
                for c in extra_cols:
                    entry[c] = row[c]
            rows.append(entry)
    return pd.DataFrame(rows)

sents_reddit = explode_sentences(df_reddit, "text", "reddit",    date_col="date",         extra_cols=["subreddit"])
sents_web    = explode_sentences(df_web,    "text", "everlane",  date_col=None,           extra_cols=["page"])
sents_news   = explode_sentences(df_news,   "text", "news",      date_col="publish_date")

df_sents = pd.concat([sents_reddit, sents_web, sents_news], ignore_index=True)
df_sents["date"] = pd.to_datetime(df_sents["date"], errors="coerce")

print(df_sents["source"].value_counts().to_string())
print(f"\nTotal sentences: {len(df_sents):,}")

# 1. Manual LLM Pipeline (Claude API)

Before scaling anything, we use Claude to develop intuitions on a small sample. The goal is twofold:
- **Validate** that our sentence-level units actually contain meaningful signal
- **Define** the classification categories we care about before handing off to BERT

We sample 8 sentences from each source and ask Claude to classify each along two dimensions:
1. **Claim type**: `concrete_claim` | `vague_marketing` | `consumer_complaint` | `media_framing` | `other`
2. **Sentiment toward Everlane**: `positive` | `neutral` | `negative`

This manual pass is intentionally small — its purpose is to give us a ground-truth intuition for what the BERT model will measure at scale.

In [ ]:
import anthropic

# Set your API key — either paste directly or set ANTHROPIC_API_KEY in environment
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

# Sample 8 sentences per source, seeded for reproducibility
rng = np.random.default_rng(42)

def sample_source(df, source, n=8):
    pool = df[df["source"] == source]["sentence"]
    # prefer sentences containing "everlane" for relevance
    everlane_pool = pool[pool.str.lower().str.contains("everlane")]
    chosen = everlane_pool.sample(min(n, len(everlane_pool)), random_state=42)
    if len(chosen) < n:
        remainder = pool[~pool.index.isin(chosen.index)].sample(n - len(chosen), random_state=42)
        chosen = pd.concat([chosen, remainder])
    return chosen.tolist()

sample_sentences = {
    "reddit":   sample_source(df_sents, "reddit"),
    "everlane": sample_source(df_sents, "everlane"),
    "news":     sample_source(df_sents, "news"),
}

# Preview the sample
for source, sents in sample_sentences.items():
    print(f"\n--- {source.upper()} ---")
    for i, s in enumerate(sents, 1):
        print(f"  {i}. {s[:120]}")

In [ ]:
SYSTEM_PROMPT = """You are an analyst studying the fashion brand Everlane's sustainability claims and public perception.

For each sentence provided, return a JSON array where each element has:
- "sentence": the original sentence (verbatim)
- "claim_type": one of: concrete_claim | vague_marketing | consumer_complaint | media_framing | other
- "sentiment": one of: positive | neutral | negative  (sentiment *toward Everlane* as a brand)
- "reasoning": one brief sentence explaining your labels

Definitions:
- concrete_claim: a specific, verifiable statement (e.g. cites a number, names a factory, describes a measurable action)
- vague_marketing: aspirational or feel-good language with no measurable commitment
- consumer_complaint: a user expressing dissatisfaction with Everlane's product, ethics, or behavior
- media_framing: a journalist contextualizing Everlane within a broader story

Return only valid JSON. No markdown fences."""

def classify_sample(sentences, source_label):
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    user_msg = f"Source: {source_label}\n\nSentences:\n{numbered}"
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )
    import json
    return json.loads(response.content[0].text)

# Run on all three sources
llm_results = {}
for source, sents in sample_sentences.items():
    print(f"Classifying {source}...")
    llm_results[source] = classify_sample(sents, source)
    print(f"  Done — {len(llm_results[source])} sentences classified")

print("\nAll done.")

In [ ]:
# Flatten results into a dataframe for inspection
llm_rows = []
for source, results in llm_results.items():
    for r in results:
        r["source"] = source
        llm_rows.append(r)

df_llm = pd.DataFrame(llm_rows)[["source", "claim_type", "sentiment", "sentence", "reasoning"]]
display(df_llm)

# Quick summary: how did Claude label each source?
print("\nClaim type distribution by source:")
display(df_llm.groupby(["source", "claim_type"]).size().unstack(fill_value=0))

print("\nSentiment distribution by source:")
display(df_llm.groupby(["source", "sentiment"]).size().unstack(fill_value=0))

# 2. Automated BERT Pipeline

Now we scale to the full corpus using two HuggingFace models:

**NER** — `dslim/bert-base-NER`  
Identifies named entities (ORG, PER, LOC) in each sentence. We use this to flag *inter-party sentences*: sentences from Reddit or news that mention Everlane (brand talked about by others), and sentences from Everlane's website that mention labor-related entities (workers, factories, partners).

**Sentiment** — `cardiffnlp/twitter-roberta-base-sentiment`  
Three-class sentiment (negative / neutral / positive). Trained on social-media text so it handles Reddit's informal register well. For news and web text the register is more formal, but for this project a single consistent model is preferable to a confusing multi-model comparison.

The pipeline runs in batches to avoid OOM. On CPU this takes a few minutes; on GPU it's much faster.

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")

ner_pipe = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=device,
)

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=device,
    truncation=True,
    max_length=512,
)

print("Models loaded.")

In [ ]:
BATCH = 64

def run_ner_batched(sentences):
    results = []
    for i in tqdm(range(0, len(sentences), BATCH), desc="NER"):
        batch = sentences[i : i + BATCH]
        out = ner_pipe(batch)
        for ents in out:
            orgs = [e["word"] for e in ents if e["entity_group"] == "ORG"]
            results.append(orgs)
    return results

def run_sentiment_batched(sentences):
    labels, scores = [], []
    for i in tqdm(range(0, len(sentences), BATCH), desc="Sentiment"):
        batch = sentences[i : i + BATCH]
        out = sentiment_pipe(batch)
        for r in out:
            # model labels: LABEL_0=negative, LABEL_1=neutral, LABEL_2=positive
            label_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
            labels.append(label_map.get(r["label"], r["label"]))
            scores.append(round(r["score"], 4))
    return labels, scores

sentences_list = df_sents["sentence"].tolist()

# NER pass
df_sents["entities"] = run_ner_batched(sentences_list)

# Sentiment pass
df_sents["sentiment"], df_sents["sentiment_score"] = run_sentiment_batched(sentences_list)

print(df_sents["sentiment"].value_counts().to_string())

## Inter-party sentence filtering

An *inter-party sentence* is one where Source A explicitly references Party B:
- **Reddit → Everlane**: consumer voices talking about the brand
- **News → Everlane**: journalists writing about the brand  
- **Everlane → labor**: the brand's own pages mentioning workers, factories, or partners

We detect these with a combination of NER entity matching and keyword fallback (NER sometimes misses "Everlane" as a brand name since it wasn't in the model's training vocabulary).

In [ ]:
LABOR_TERMS = re.compile(
    r"\b(worker|workers|employee|employees|staff|factory|factories|"
    r"supplier|suppliers|partner|partners|labour|labor|wage|wages|"
    r"artisan|artisans|maker|makers|seamstress|seamstresses)\b",
    re.IGNORECASE,
)

def mentions_everlane(row):
    text_hit = "everlane" in row["sentence"].lower()
    ner_hit  = any("everlane" in e.lower() for e in (row["entities"] or []))
    return text_hit or ner_hit

def mentions_labor(row):
    return bool(LABOR_TERMS.search(row["sentence"]))

df_sents["mentions_everlane"] = df_sents.apply(mentions_everlane, axis=1)
df_sents["mentions_labor"]    = df_sents.apply(mentions_labor, axis=1)

# Three inter-party subsets
reddit_about_everlane   = df_sents[(df_sents["source"] == "reddit")   & df_sents["mentions_everlane"]]
news_about_everlane     = df_sents[(df_sents["source"] == "news")     & df_sents["mentions_everlane"]]
everlane_about_labor    = df_sents[(df_sents["source"] == "everlane") & df_sents["mentions_labor"]]

print(f"Reddit → Everlane:  {len(reddit_about_everlane):,} sentences")
print(f"News   → Everlane:  {len(news_about_everlane):,} sentences")
print(f"Everlane → labor:   {len(everlane_about_labor):,} sentences")

# 3. Analysis

## 3a. Sentiment distribution by source

How does the overall emotional tone differ across the three voices?

In [ ]:
ORDER = ["negative", "neutral", "positive"]
PALETTE = {"negative": "#d95f02", "neutral": "#7570b3", "positive": "#1b9e77"}

dist = (
    df_sents.groupby(["source", "sentiment"])
    .size()
    .reset_index(name="count")
)
dist["pct"] = dist.groupby("source")["count"].transform(lambda x: x / x.sum() * 100)

fig, ax = plt.subplots(figsize=(8, 4))
sources = ["reddit", "news", "everlane"]
x = np.arange(len(sources))
width = 0.25

for i, sentiment in enumerate(ORDER):
    vals = [dist[(dist["source"] == s) & (dist["sentiment"] == sentiment)]["pct"].values
            for s in sources]
    vals = [v[0] if len(v) else 0 for v in vals]
    bars = ax.bar(x + i * width, vals, width, label=sentiment, color=PALETTE[sentiment])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.0f}%", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(["Reddit (consumers)", "News (media)", "Everlane (brand)"])
ax.set_ylabel("% of sentences")
ax.set_title("Sentence-level sentiment distribution by source")
ax.legend(title="Sentiment")
plt.tight_layout()
plt.show()

## 3b. Inter-party valence comparison

The key question: when each source explicitly mentions Everlane (or labor, for the brand's own pages), what is the sentiment of those targeted sentences?

This is more informative than overall source sentiment, because it isolates the *signal* — sentences that are actually about the relationship between the parties.

In [ ]:
interparty_frames = {
    "Reddit → Everlane":  reddit_about_everlane,
    "News → Everlane":    news_about_everlane,
    "Everlane → labor":   everlane_about_labor,
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

for ax, (label, frame) in zip(axes, interparty_frames.items()):
    counts = frame["sentiment"].value_counts().reindex(ORDER, fill_value=0)
    pcts   = counts / counts.sum() * 100
    colors = [PALETTE[s] for s in ORDER]
    bars   = ax.bar(ORDER, pcts, color=colors, edgecolor="white")
    for bar, val in zip(bars, pcts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.0f}%", ha="center", va="bottom", fontsize=9)
    ax.set_title(label)
    ax.set_xlabel("Sentiment")
    ax.set_ylim(0, 100)
    n = len(frame)
    ax.text(0.97, 0.97, f"n={n:,}", transform=ax.transAxes,
            ha="right", va="top", fontsize=8, color="gray")

axes[0].set_ylabel("% of inter-party sentences")
fig.suptitle("Inter-party valence: sentiment of sentences that cross party lines", y=1.02)
plt.tight_layout()
plt.show()

## 3c. Consumer sentiment over time

Has Reddit's tone toward Everlane shifted over the years? We use a 90-day rolling average of the sentiment score (0 = negative, 0.5 = neutral, 1 = positive) on inter-party sentences only.

In [ ]:
# Map label to a numeric score for rolling average
SCORE_MAP = {"negative": 0.0, "neutral": 0.5, "positive": 1.0}

def rolling_sentiment(frame, label, color, ax, window="90D"):
    ts = frame.dropna(subset=["date"]).copy()
    ts["score_num"] = ts["sentiment"].map(SCORE_MAP)
    ts = ts.set_index("date").sort_index()["score_num"]
    rolled = ts.rolling(window, min_periods=5).mean()
    rolled.plot(ax=ax, label=label, color=color, linewidth=1.5, alpha=0.9)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Reddit consumer sentiment toward Everlane
rolling_sentiment(reddit_about_everlane, "Reddit → Everlane", "#e07b39", axes[0])
axes[0].set_title("Consumer sentiment toward Everlane over time (Reddit, 90-day rolling avg)")
axes[0].set_ylabel("Sentiment score")
axes[0].set_ylim(0, 1)
axes[0].legend()

# News media sentiment toward Everlane
rolling_sentiment(news_about_everlane, "News → Everlane", "#3a86ff", axes[1])
axes[1].set_title("Media sentiment toward Everlane over time (News, 90-day rolling avg)")
axes[1].set_ylabel("Sentiment score")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

for ax in axes:
    ax.text(0.01, 0.05, "← more negative", transform=ax.transAxes, fontsize=8, color="gray")
    ax.text(0.01, 0.90, "← more positive", transform=ax.transAxes, fontsize=8, color="gray")

plt.tight_layout()
plt.show()

# 4. LLM Synthesis — Naming the Clusters

BERT gave us sentiment scores across the full corpus. Now we bring Claude back in to answer one question BERT can't: **what are people actually saying** in the most negative and most positive inter-party sentences?

We pull the top-N high-confidence sentences from each polarity bucket, group them by source, and ask Claude to name the dominant themes. This is a one-time synthesis call — not a loop over every sentence — so it's cheap and fast.

In [ ]:
def top_sentences(frame, sentiment_label, n=20):
    """Return the n highest-confidence sentences for a given sentiment label."""
    subset = frame[frame["sentiment"] == sentiment_label].nlargest(n, "sentiment_score")
    return subset["sentence"].tolist()

clusters = {
    "Reddit — most negative about Everlane":  top_sentences(reddit_about_everlane, "negative"),
    "Reddit — most positive about Everlane":  top_sentences(reddit_about_everlane, "positive"),
    "News — most negative about Everlane":    top_sentences(news_about_everlane,   "negative"),
    "News — most positive about Everlane":    top_sentences(news_about_everlane,   "positive"),
    "Everlane website — labor language":      everlane_about_labor["sentence"].tolist()[:20],
}

SYNTHESIS_SYSTEM = """You are a brand analyst. Given a cluster of sentences from a specific source, 
identify the 2-3 dominant themes. For each theme give:
- a short label (3-5 words)
- one representative quote from the sentences provided
- one sentence explaining what this theme reveals about Everlane's reputation or messaging

Be concrete. Reference specific language from the sentences. Return plain text, not JSON."""

synthesis_results = {}
for cluster_name, sents in clusters.items():
    if not sents:
        print(f"SKIP (empty): {cluster_name}")
        continue
    print(f"Synthesizing: {cluster_name}...")
    numbered = "\n".join(f"- {s}" for s in sents)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system=SYNTHESIS_SYSTEM,
        messages=[{"role": "user", "content": f"Cluster: {cluster_name}\n\nSentences:\n{numbered}"}],
    )
    synthesis_results[cluster_name] = response.content[0].text

print("\nDone.")

In [ ]:
for cluster_name, analysis in synthesis_results.items():
    print(f"\n{'='*70}")
    print(f"  {cluster_name}")
    print(f"{'='*70}")
    print(analysis)

# 5. Conclusions

## What the two pipelines together tell us

**Manual LLM pass (Section 1)**  
Claude's classifications on the hand-picked sample establish the vocabulary of this corpus: what a "concrete claim" looks like in Everlane's own language vs. what "consumer complaint" looks like on Reddit. These qualitative anchors give us a frame of reference for interpreting the BERT scores.

**Automated BERT pass (Sections 2–3)**  
Running NER and sentiment across all ~47,000 sentences surfaces a quantitative answer to the research question. The inter-party valence charts in Section 3b are the key result: they show whether the *brand's own tone toward labor* aligns or diverges from the *consumers' tone toward the brand*.

**LLM synthesis (Section 4)**  
Rather than reading thousands of negative sentences manually, we sent the high-confidence extremes back to Claude to name the themes. This closes the loop — the automation found the most interesting examples, and the LLM gave them interpretable labels.

## Limitations
- The BERT model was trained on social media text; its register calibration on formal web copy (Everlane pages) is imperfect
- GDELT news coverage skews toward recent years, so the time-series comparison is not perfectly balanced
- Inter-party filtering via keyword regex may miss paraphrased references and over-include tangential mentions